# Fundamentos de Bases de Datos y SQL en Databricks
# 1.1 ¿Qué es una Base de Datos?
Una base de datos es una colección organizada de información o datos estructurados que normalmente se almacenan electrónicamente en un sistema informático. Permite almacenar, consultar, actualizar y gestionar datos de forma eficiente.

**Tipos de Bases de Datos: Relacionales vs. NoSQL**

_**Bases de Datos Relacionales (RDBMS):**_ Organizan los datos en tablas compuestas por filas y columnas con esquemas rígidos y relaciones predefinidas entre ellas (ej. PostgreSQL, MySQL, SQL Server).

**_Bases de Datos NoSQL (No Relacionales):_** Diseñadas para modelos de datos específicos (documentos, clave-valor, grafos, columnas anchas) con esquemas flexibles e ideal para grandes volúmenes de datos no estructurados (ej. MongoDB, Cassandra, Neo4j).

**Databricks combina lo mejor de ambos mundos mediante Delta Lake.** Utiliza un enfoque Lakehouse, permitiendo ejecutar consultas SQL con semántica y garantías transaccionales (ACID) propias de las bases de datos relacionales directamente sobre archivos masivos de datos almacenados en la nube.

## 1.2 El Modelo Relacional
El modelo relacional organiza la información en entidades estructuradas:

**_Tabla:_** La unidad principal de almacenamiento. Representa un conjunto de entidades homogéneas (ej. clientes, ventas).

**_Fila (Registro / Tupla):_** Cada entrada individual dentro de una tabla. Representa un único elemento o evento.

**_Columna (Campo / Atributo):_** Cada propiedad o característica que define a las filas.

## 1.3 Tipos de Datos Comunes en Databricks SQL
En Databricks SQL (basado en Spark SQL), los tipos de datos principales incluyen:

**_INT / INTEGER_** : Números enteros de 32 bits (ej. 42, -100).

**_STRING:_** Cadenas de texto de longitud variable (reemplaza al VARCHAR tradicional de SQL).

**_DATE_**: Fechas en formato AAAA-MM-DD (ej. '2026-10-28').

**_TIMESTAMP:_** Fecha y hora exacta con precisión de microsegundos (ej. '2026-10-28 14:30:00').

**BOOLEAN:** Valores lógicos de verdad (TRUE o FALSE).

**_DECIMAL(p, s) / NUMERIC(p, s)_**: Números de precisión exacta (útil para datos financieros; ej. DECIMAL(10, 2) admite hasta 10 dígitos, 2 de ellos decimales).

**La funcion CAST permite asignar el tipo de dato esperado para cada variable para garantizar tipado adecuado para el análisis**

In [0]:
%sql

-- CASTEANDO CADA COLUMNA A SU TIPO DE DATO ESPERADO
SELECT 
    CAST(id_transaccion AS STRING),
    CAST(fecha AS DATE),
    CAST(id_cliente AS STRING),
    CAST(categoria_producto AS STRING),
    CAST(cantidad AS DOUBLE),
    CAST(precio_unitario AS DOUBLE),
    CAST(metodo_pago AS STRING),
    CAST(region AS STRING),
    CAST(satisfaccion_cliente AS DOUBLE)
FROM workspace.electiva.caso_practico_ventas 

## 1.4 Claves Primarias, Claves Foráneas e Integridad
**_Clave Primaria (Primary Key - PK):_** Columna o conjunto de columnas que identifica de forma única a cada fila de una tabla. No puede contener valores duplicados ni nulos.

**_Clave Foránea (Foreign Key - FK):_** Columna en una tabla que hace referencia a la Clave Primaria de otra tabla, creando una relación o vínculo directo entre ambas.


![image_1793199487083.png](./image_1793199487083.png "image_1793199487083.png")

## 1.5 El Concepto de NULL
En SQL, NULL no significa cero (0) ni una cadena vacía ('').

NULL representa la ausencia de valor o dato desconocido.

**Reglas básicas al trabajar con NULL:**

- _Cualquier operación aritmética con NULL da como resultado NULL (ej. 5 + NULL = NULL)._

- _Para comparar un valor con NULL, no se debe usar = NULL o <> NULL. En su lugar se utiliza IS NULL o IS NOT NULL_

In [0]:
%sql
select 
    * 
from workspace.electiva.caso_practico_ventas
where cantidad = null; -- al comparar cantidad = null no aparece ningun registro, lo cual es un resultado incorrecto


In [0]:
%sql
select 
    * 
from workspace.electiva.caso_practico_ventas
where cantidad is  null; -- al comparar cantidad  is null nos muestra los registros que no tienen un valor en la columna cantidad, que es el resultado esperado correcto

##  1.6 Alias
_Un Alias es un nombre temporal y alternativo que le asignas a una columna, a una expresión calculada o a una tabla dentro de una consulta._

Los alias no cambian los nombres reales de las tablas ni de las columnas en el almacenamiento físico (como tus tablas Delta Lake); solo existen durante el tiempo que tarda en ejecutarse la consulta para cambiar cómo se muestra el resultado o para facilitar la escritura del código.

**¿Para qué sirven los Alias?**

**_Legibilidad y Limpieza:_** Permiten renombrar columnas calculadas o complejas por nombres simples y entendibles (ej. reemplazar SUM(monto_total * 0.19) por impuesto).

**_Evitar Ambigüedad_**: Cuando haces cruces (JOINs) entre varias tablas que tienen columnas con el mismo nombre (ej. id), los alias te ayudan a indicar exactamente de qué tabla proviene cada dato.

_**Productividad**_: Reducen la cantidad de texto que debes escribir al usar nombres cortos para las tablas.

In [0]:

%sql
-- USAMOS LA SINTAXIS "AS" para ASIGNAR UN ALIAS a las columnas
SELECT 
    id_transaccion  AS codigo_transaccion, -- se renombra la columna id_transaccion -> codigo_transaccion
    fecha as fecha_transaccion, --se renombra la columna fecha -> fecha_transaccion
    id_cliente as cedula_cliente, --se renombra la columna id_cliente -> cedula_cliente
    categoria_producto ,
    cantidad ,
    precio_unitario ,
    metodo_pago ,
    region ,
    satisfaccion_cliente
FROM workspace.electiva.caso_practico_ventas;

In [0]:

%sql
-- USAMOS LA SINTAXIS "AS" para ASIGNAR UN ALIAS a las tablas que estamos consultando
SELECT 
    ventas.id_transaccion  AS codigo_transaccion, -- se renombra la columna id_transaccion -> codigo_transaccion
    ventas.fecha as fecha_transaccion, --se renombra la columna fecha -> fecha_transaccion
    ventas.id_cliente as cedula_cliente, --se renombra la columna id_cliente -> cedula_cliente
    ventas.categoria_producto ,
    ventas.cantidad ,
    ventas.precio_unitario ,
    ventas.metodo_pago ,
    ventas.region ,
    ventas.satisfaccion_cliente
FROM workspace.electiva.caso_practico_ventas as ventas; -- asignamos un alias a la tabla caso_practico_ventas -> ventas

##  1.7 SQL y Dialectos
**SQL (Structured Query Language)** es el lenguaje estándar del dominio para interactuar con bases de datos relacionales.

**ANSI SQL:** El estándar internacional definido para el lenguaje SQL.

**Dialectos SQL:** Variaciones del estándar que implementa cada motor comercial para añadir funciones específicas u optimizaciones propias (PostgreSQL, MySQL, T-SQL, PL/SQL).

**Databricks SQL (Spark SQL):** Dialecto diseñado para procesamiento masivo distribuido. Conserva gran parte de la sintaxis estándar ANSI SQL, pero añade optimizaciones y funciones avanzadas para analizar Big Data y gestionar tablas Delta

##  1.8 Creacion de tablas SQL 
##  

Para crear tablas en databricks con SQL usamos la sintaxis

 `CREATE TABLE IF NOT EXISTS <nombre_tabla>`

**Cláusulas más importantes:**

`IF NOT EXISTS`: Evita que la consulta falle si la tabla ya fue creada previamente.

`COMMENT`: Permite añadir documentación directamente en las celdas o columnas.

`TBLPROPERTIES`: Permite configurar propiedades avanzadas del motor Delta (como habilitar eliminación lógica o auto-optimización).

In [0]:
%sql
-- 1. Crear tabla de Clientes
CREATE TABLE IF NOT EXISTS clientes (
    cliente_id INT,
    nombre STRING,
    email STRING,
    telefono STRING, -- Contiene NULLs
    fecha_registro STRING -- Formato texto 'YYYY-MM-DD' para practicar casteo
) USING DELTA
COMMENT 'Tabla para almacenar informacion de clientes';

-- 2. Crear tabla de Productos
CREATE TABLE IF NOT EXISTS productos (
    producto_id INT,
    nombre_producto STRING,
    categoria STRING,
    precio STRING, -- Almacenado como STRING para practicar CAST a numérico
    descuento_pct DECIMAL(4,2) -- Contiene NULLs cuando no hay descuento
) USING DELTA
COMMENT 'Tabla para almacenar informacion de productos';

-- 3. Crear tabla de Pedidos
CREATE TABLE IF NOT EXISTS pedidos (
    pedido_id INT,
    cliente_id INT,
    fecha_pedido STRING, -- Formato texto 'YYYY-MM-DD HH:MM:SS' para CAST a TIMESTAMP
    estado STRING
) USING DELTA
COMMENT 'TABLA PARA ALMACENAR INFORMACION DE PEDIDOS';

-- 4. Crear tabla de Detalles del Pedido
CREATE TABLE IF NOT EXISTS detalles_pedido (
    detalle_id INT,
    pedido_id INT,
    producto_id INT,
    cantidad INT
) USING DELTA
COMMENT 'TABLA PARA ALMACENAR INFORMACION DE DETALLES DEL PEDIDO';

##  1.9 Eliminacion de tablas SQL 
Para eliminar tablas , usamos el comando `DROP TABLE <nombre_tabla> `

In [0]:
%sql
DROP TABLE IF EXISTS workspace.default.clientes;
DROP TABLE IF EXISTS workspace.default.productos;
DROP TABLE IF EXISTS workspace.default.pedidos;
DROP TABLE IF EXISTS workspace.default.detalles_pedido;


## 2.0 Recrear tablas en el esquema correcto

In [0]:
%sql
-- 1. Crear tabla de Clientes
CREATE TABLE IF NOT EXISTS workspace.electiva.clientes (
    cliente_id INT,
    nombre STRING,
    email STRING,
    telefono STRING, -- Contiene NULLs
    fecha_registro STRING -- Formato texto 'YYYY-MM-DD' para practicar casteo
) USING DELTA
COMMENT 'Tabla para almacenar informacion de clientes';

-- 2. Crear tabla de Productos
CREATE TABLE IF NOT EXISTS workspace.electiva.productos (
    producto_id INT,
    nombre_producto STRING,
    categoria STRING,
    precio STRING, -- Almacenado como STRING para practicar CAST a numérico
    descuento_pct DECIMAL(4,2) -- Contiene NULLs cuando no hay descuento
) USING DELTA
COMMENT 'Tabla para almacenar informacion de productos';

-- 3. Crear tabla de Pedidos
CREATE TABLE IF NOT EXISTS workspace.electiva.pedidos (
    pedido_id INT,
    cliente_id INT,
    fecha_pedido STRING, -- Formato texto 'YYYY-MM-DD HH:MM:SS' para CAST a TIMESTAMP
    estado STRING
) USING DELTA
COMMENT 'TABLA PARA ALMACENAR INFORMACION DE PEDIDOS';

-- 4. Crear tabla de Detalles del Pedido
CREATE TABLE IF NOT EXISTS workspace.electiva.detalles_pedido (
    detalle_id INT,
    pedido_id INT,
    producto_id INT,
    cantidad INT
) USING DELTA
COMMENT 'TABLA PARA ALMACENAR INFORMACION DE DETALLES DEL PEDIDO';